# Pooling Generation (Qrels)
This notebook generates the evaluation pool by extracting the top results from BM25, Dense, and Hybrid search for a set of test queries.

In [8]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
from src.search_engine import BM25Searcher, DenseSearcher, HybridSearcher

# 50 TEST QUERIES CATEGORIZED 
TEST_QUERIES = [
    # Tramas, conceptos y emociones sin nombres propios
    "A story of survival in extreme cold weather and snow",
    "Psychological tension and family secrets between relatives",
    "High school students forming a music band and dealing with youth problems",
    "A journey through outer space to find a new habitable planet for humanity",
    "Documentary showing the history of jazz music and its cultural impact",
    "A detective dealing with his own inner demons while chasing a serial killer",
    "Romantic relationship between people from completely different social classes",
    "A struggle against an oppressive dystopian government in a near future",
    "Feel-good comedy about an unexpected friendship between a senior and a teenager",
    "Athletes overcoming severe physical injuries to win a championship",
    "A house haunted by malevolent spirits that terrorize a family",
    "Time travel causing complex paradoxes and altering historical events",
    "An deep dive into the lives of wild animals in the African savannah",
    "A lawyer fighting against a corrupt corporation to protect poor citizens",
    "An artist struggling with writer's block and losing their mind",
    "A classic revenge story where a protagonist hunts down former betrayers",
    "Coming of age story of a young boy finding his identity in a small town",

    # Directores, actores, combinaciones
    "Movies directed by Steven Spielberg",
    "Movies directed by Martin Scorsese",
    "Films starring Samuel L. Jackson",
    "Movies starring Danny Trejo",
    "Films starring Shah Rukh Khan",
    "Movies starring Amitabh Bachchan",
    "Films starring Nicolas Cage",
    "Movies starring Bruce Willis",
    "Movies directed by Robert Vince",
    "Films directed by Paul Hoen",
    "Movies starring Anupam Kher",
    "Films starring Maggie Binkley",
    "Movies starring Paresh Rawal",
    "Films starring Naseeruddin Shah",
    "Movies starring Akshay Kumar",
    "Films directed by John Lasseter",
    "Movies starring Om Puri",

    # Entidad + Concepto
    "Action movies starring Samuel L. Jackson",
    "Science fiction movies directed by Steven Spielberg",
    "Animated films directed by John Lasseter",
    "Documentaries featuring Elon Musk",
    "Drama movies starring Nicolas Cage",
    "Romantic comedy dramas starring Shah Rukh Khan",
    "Horror movies starring Danny Trejo",
    "Crime drama thriller directed by Martin Scorsese",
    "Comedy movies starring Bruce Willis",
    "Action thrillers starring Amitabh Bachchan",
    "Sports documentary starring or about athletes",
    "Family comedy directed by Robert Vince",
    "Musical comedy directed by Paul Hoen",
    "Comedy movies starring Anupam Kher",
    "Drama movies starring Naseeruddin Shah",
    "Action movies starring Akshay Kumar",

    "A science fiction movie about space exploration starring Matthew McConaughey",
    "A gritty crime thriller about a serial killer starring Anthony Hopkins",
    "Romantic comedy set in New York directed by Nora Ephron",
    "Action movie involving a bank robbery or hostage situation starring Keanu Reeves",
    "A coming of age drama in a small town starring Timothée Chalamet",
    "A fantasy movie about magic, elves or wizards directed by Peter Jackson",
    "Documentary about nature, animals and the ocean narrated by David Attenborough",
    "A dramatic war movie set during World War II directed by Quentin Tarantino",
    "A feel-good family movie about dogs or pets starring Paul Walker",
    "A suspenseful mystery or psychological thriller starring Leonardo DiCaprio"
]

In [9]:
# Load the dataframe with embeddings
print("Loading dataframe...")
movies = pd.read_pickle('../data/movies_with_embeddings.pkl')

# Initialize searchers
bm25_searcher = BM25Searcher(movies)
dense_searcher = DenseSearcher(movies)
hybrid_searcher = HybridSearcher(bm25_searcher, dense_searcher)


Loading dataframe...
Loading precalculated BM25 index from /teamspace/studios/this_studio/semantic-movie-retrieval-nlp/data/bm25_index.pkl...
BM25 Index loaded successfully.
Loading SentenceTransformer model (multi-qa-MiniLM-L6-cos-v1)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading precalculated FAISS index from /teamspace/studios/this_studio/semantic-movie-retrieval-nlp/data/faiss_index.bin...
FAISS Index and ID map loaded successfully.


In [10]:
# Generate Pool
pool_rows = []
top_k = 10
global_id = 0

for query in TEST_QUERIES:
    print(f"Processing query: '{query}'")
    bm25_res = bm25_searcher.search(query, top_k=top_k)
    dense_res = dense_searcher.search(query, top_k=top_k)
    hybrid_res = hybrid_searcher.search(query, top_k=top_k)
    
    unique_titles = set()
    for res in bm25_res + dense_res + hybrid_res:
        unique_titles.add(res['title'])
        
    for title in sorted(unique_titles):
        pool_rows.append({
            'id': global_id,
            'query': query,
            'movie_title': title,
            'relevance': ''
        })
        global_id += 1

pool_df = pd.DataFrame(pool_rows)
print(f"\nGenerated pool with {len(pool_df)} unique query-movie pairs.")

# Formatter for copy-pasting to the LLM 
movie_desc_dict = dict(zip(movies['title'], movies['content_narrative']))

prompt_content = []
prompt_header = """Instructions for the AI Labeler:
You are an expert movie relevance annotator.
For each item below, you are given a unique ID, a Search Query, a Movie Title, and a Narrative Content (synopsis and details).
Grade the relevance of the movie to the query on a scale of 0 to 5:
0: Completely irrelevant (no connection whatsoever)
1: Barely relevant (very weak or accidental keyword overlap)
2: Somewhat relevant (shares some elements like genre or minor themes, but not matching core intent)
3: Relevant (matches the main concepts/entities of the search query, but might miss specific aspects)
4: Highly relevant (excellent match to the query, covers the search intent very well)
5: Perfect match (exact match to the query, or is the precise movie/franchise/subject requested)

Return your response as a simple list where each line has the format:
ID: Score
Example:
0: 5
1: 0
2: 3

Here are the items to grade:
"""
prompt_content.append(prompt_header)

for _, row in pool_df.iterrows():
    row_id = row['id']
    q = row['query']
    title = row['movie_title']
    desc = movie_desc_dict.get(title, "No description available.")
    
    item_str = f"ID: {row_id}\nQuery: {q}\nMovie Title: {title}\nNarrative Content: {desc}\n--------------------------------------------------\n"
    prompt_content.append(item_str)

output_prompt_path = '../data/prompt_for_ai.txt'
with open(output_prompt_path, 'w', encoding='utf-8') as f:
    f.writelines(prompt_content)

print(f"AI Prompt successfully exported to '{output_prompt_path}'.")
print("Preview of the first 3 items:")
print("".join(prompt_content[1:4]))

Processing query: 'A story of survival in extreme cold weather and snow'
Processing query: 'Psychological tension and family secrets between relatives'
Processing query: 'High school students forming a music band and dealing with youth problems'
Processing query: 'A journey through outer space to find a new habitable planet for humanity'
Processing query: 'Documentary showing the history of jazz music and its cultural impact'
Processing query: 'A detective dealing with his own inner demons while chasing a serial killer'
Processing query: 'Romantic relationship between people from completely different social classes'
Processing query: 'A struggle against an oppressive dystopian government in a near future'
Processing query: 'Feel-good comedy about an unexpected friendship between a senior and a teenager'
Processing query: 'Athletes overcoming severe physical injuries to win a championship'
Processing query: 'A house haunted by malevolent spirits that terrorize a family'
Processing query

In [11]:
# Results Importer
import re
import os

ai_output_raw = """
0: 3
1: 5
2: 1
3: 1
4: 3
5: 1
6: 2
7: 3
8: 3
9: 2
10: 0
11: 2
12: 2
13: 2
14: 2
15: 2
16: 1
17: 3
18: 2
19: 2
20: 2
21: 3
22: 3
23: 4
24: 3
25: 1
26: 3
27: 1
28: 3
29: 2
30: 0
31: 3
32: 2
33: 4
34: 4
35: 2
36: 1
37: 3
38: 2
39: 2
40: 5
41: 2
42: 2
43: 2
44: 3
45: 2
46: 2
47: 2
48: 1
49: 2
50: 1
51: 1
52: 2
53: 1
54: 2
55: 2
56: 3
57: 2
58: 2
59: 1
60: 3
61: 2
62: 2
63: 3
64: 1
65: 2
66: 3
67: 2
68: 2
69: 2
70: 1
71: 2
72: 3
73: 1
74: 2
75: 3
76: 4
77: 2
78: 1
79: 1
80: 2
81: 3
82: 4
83: 1
84: 3
85: 2
86: 0
87: 2
88: 1
89: 3
90: 1
91: 3
92: 2
93: 2
94: 4
95: 2
96: 2
97: 1
98: 1
99: 4
100: 3
101: 3
102: 2
103: 4
104: 3
105: 4
106: 3
107: 4
108: 2
109: 4
110: 1
111: 4
112: 2
113: 4
114: 4
115: 4
116: 3
117: 2
118: 2
119: 2
120: 2
121: 2
122: 3
123: 4
124: 5
125: 2
126: 3
127: 2
128: 5
129: 2
130: 1
131: 2
132: 4
133: 2
134: 1
135: 4
136: 4
137: 2
138: 4
139: 4
140: 1
141: 4
142: 1
143: 2
144: 1
145: 2
146: 3
147: 2
148: 3
149: 2
150: 2
151: 4
152: 2
153: 3
154: 4
155: 3
156: 2
157: 2
158: 4
159: 2
160: 1
161: 2
162: 2
163: 2
164: 2
165: 2
166: 1
167: 1
168: 2
169: 2
170: 2
171: 2
172: 2
173: 2
174: 2
175: 3
176: 1
177: 3
178: 2
179: 2
180: 1
181: 2
182: 1
183: 2
184: 1
185: 2
186: 3
187: 1
188: 2
189: 1
190: 2
191: 2
192: 2
193: 3
194: 2
195: 2
196: 3
197: 3
198: 3
199: 4
200: 4
201: 2
202: 3
203: 3
204: 1
205: 2
206: 3
207: 3
208: 3
209: 4
210: 3
211: 3
212: 2
213: 5
214: 4
215: 4
216: 5
217: 4
218: 3
219: 1
220: 1
221: 3
222: 1
223: 1
224: 2
225: 2
226: 4
227: 2
228: 4
229: 3
230: 3
231: 3
232: 4
233: 4
234: 1
235: 1
236: 3
237: 2
238: 2
239: 2
240: 5
241: 2
242: 4
243: 2
244: 2
245: 1
246: 2
247: 4
248: 1
249: 2
250: 2
251: 2
252: 3
253: 1
254: 1
255: 1
256: 4
257: 2
258: 2
259: 3
260: 2
261: 2
262: 2
263: 2
264: 3
265: 3
266: 3
267: 3
268: 1
269: 4
270: 1
271: 3
272: 3
273: 4
274: 4
275: 2
276: 5
277: 2
278: 4
279: 3
280: 3
281: 3
282: 2
283: 3
284: 1
285: 2
286: 3
287: 3
288: 2
289: 3
290: 2
291: 3
292: 3
293: 2
294: 4
295: 3
296: 3
297: 3
298: 3
299: 4
300: 3
301: 3
302: 2
303: 3
304: 3
305: 2
306: 2
307: 4
308: 4
309: 3
310: 3
311: 4
312: 3
313: 2
314: 3
315: 5
316: 3
317: 3
318: 3
319: 4
320: 3
321: 4
322: 3
323: 3
324: 4
325: 4
326: 3
327: 2
328: 3
329: 2
330: 4
331: 2
332: 3
333: 2
334: 2
335: 5
336: 0
337: 5
338: 5
339: 5
340: 5
341: 5
342: 0
343: 5
344: 0
345: 5
346: 5
347: 5
348: 0
349: 5
350: 5
351: 0
352: 0
353: 5
354: 5
355: 1
356: 0
357: 5
358: 5
359: 5
360: 5
361: 1
362: 0
363: 5
364: 5
365: 5
366: 0
367: 5
368: 4
369: 1
370: 2
371: 0
372: 0
373: 1
374: 1
375: 1
376: 1
377: 2
378: 1
379: 4
380: 1
381: 1
382: 4
383: 0
384: 2
385: 4
386: 2
387: 1
388: 1
389: 1
390: 1
391: 2
392: 5
393: 5
394: 2
395: 2
396: 2
397: 2
398: 2
399: 1
400: 0
401: 0
402: 0
403: 2
404: 2
405: 2
406: 1
407: 2
408: 1
409: 1
410: 3
411: 5
412: 0
413: 1
414: 5
415: 2
416: 1
417: 5
418: 2
419: 1
420: 0
421: 4
422: 2
423: 2
424: 1
425: 2
426: 0
427: 5
428: 3
429: 3
430: 5
431: 3
432: 1
433: 1
434: 2
435: 2
436: 5
437: 5
438: 1
439: 3
440: 3
441: 1
442: 5
443: 1
444: 1
445: 1
446: 1
447: 5
448: 5
449: 1
450: 5
451: 1
452: 0
453: 2
454: 0
455: 5
456: 5
457: 1
458: 0
459: 5
460: 1
461: 1
462: 5
463: 1
464: 5
465: 1
466: 1
467: 1
468: 5
469: 5
470: 5
471: 1
472: 1
473: 1
474: 1
475: 1
476: 2
477: 5
478: 5
479: 5
480: 1
481: 1
482: 5
483: 5
484: 5
485: 5
486: 1
487: 1
488: 2
489: 1
490: 5
491: 5
492: 5
493: 2
494: 1
495: 5
496: 5
497: 5
498: 5
499: 5
500: 2
501: 5
502: 5
503: 1
504: 2
505: 1
506: 5
507: 5
508: 1
509: 5
510: 1
511: 1
512: 1
513: 1
514: 5
515: 1
516: 5
517: 5
518: 1
519: 1
520: 2
521: 5
522: 5
523: 5
524: 2
525: 5
526: 5
527: 1
528: 2
529: 3
530: 1
531: 1
532: 1
533: 1
534: 5
535: 1
536: 5
537: 1
538: 2
539: 1
540: 2
541: 2
542: 5
543: 2
544: 2
545: 1
546: 2
547: 1
548: 2
549: 1
550: 1
551: 1
552: 2
553: 3
554: 3
555: 3
556: 3
557: 1
558: 1
559: 2
560: 1
561: 3
562: 2
563: 2
564: 1
565: 1
566: 1
567: 2
568: 1
569: 1
570: 1
571: 1
572: 1
573: 4
574: 1
575: 1
576: 3
577: 5
578: 2
579: 2
580: 2
581: 5
582: 1
583: 1
584: 5
585: 2
586: 1
587: 1
588: 2
589: 1
590: 1
591: 2
592: 1
593: 3
594: 1
595: 1
596: 3
597: 2
598: 2
599: 1
600: 1
601: 2
602: 3
603: 2
604: 2
605: 2
606: 1
607: 5
608: 5
609: 5
610: 1
611: 1
612: 2
613: 2
614: 1
615: 5
616: 5
617: 2
618: 2
619: 1
620: 2
621: 2
622: 1
623: 2
624: 1
625: 5
626: 5
627: 5
628: 0
629: 1
630: 5
631: 5
632: 5
633: 5
634: 5
635: 5
636: 0
637: 1
638: 5
639: 5
640: 0
641: 5
642: 0
643: 5
644: 5
645: 1
646: 2
647: 2
648: 2
649: 2
650: 2
651: 2
652: 2
653: 2
654: 4
655: 1
656: 2
657: 1
658: 1
659: 1
660: 3
661: 3
662: 3
663: 1
664: 4
665: 2
666: 1
667: 1
668: 2
669: 1
670: 1
671: 3
672: 3
673: 2
674: 2
675: 1
676: 4
677: 1
678: 4
679: 3
680: 4
681: 1
682: 1
683: 1
684: 1
685: 2
686: 3
687: 3
688: 5
689: 4
690: 4
691: 3
692: 1
693: 4
694: 2
695: 5
696: 1
697: 3
698: 2
699: 2
700: 2
701: 3
702: 2
703: 5
704: 1
705: 5
706: 5
707: 5
708: 5
709: 5
710: 5
711: 5
712: 4
713: 2
714: 5
715: 2
716: 5
717: 1
718: 5
719: 5
720: 1
721: 2
722: 2
723: 2
724: 1
725: 5
726: 1
727: 2
728: 2
729: 2
730: 4
731: 1
732: 1
733: 2
734: 2
735: 2
736: 1
737: 2
738: 2
739: 3
740: 3
741: 5
742: 1
743: 3
744: 5
745: 1
746: 2
747: 5
748: 2
749: 2
750: 5
751: 3
752: 2
753: 5
754: 2
755: 5
756: 3
757: 3
758: 2
759: 2
760: 3
761: 2
762: 4
763: 4
764: 1
765: 3
766: 4
767: 3
768: 2
769: 4
770: 4
771: 3
772: 2
773: 3
774: 4
775: 2
776: 2
777: 3
778: 2
779: 2
780: 2
781: 4
782: 4
783: 2
784: 2
785: 2
786: 1
787: 2
788: 3
789: 3
790: 2
791: 2
792: 3
793: 3
794: 3
795: 4
796: 2
797: 3
798: 5
799: 5
800: 2
801: 4
802: 2
803: 2
804: 2
805: 4
806: 1
807: 5
808: 5
809: 5
810: 2
811: 4
812: 4
813: 3
814: 3
815: 2
816: 4
817: 3
818: 2
819: 1
820: 1
821: 2
822: 2
823: 4
824: 2
825: 4
826: 4
827: 1
828: 4
829: 4
830: 4
831: 4
832: 3
833: 3
834: 3
835: 2
836: 5
837: 3
838: 2
839: 5
840: 3
841: 2
842: 2
843: 2
844: 2
845: 2
846: 2
847: 4
848: 2
849: 2
850: 4
851: 2
852: 4
853: 4
854: 3
855: 3
856: 4
857: 4
858: 4
859: 4
860: 5
861: 4
862: 5
863: 2
864: 2
865: 2
866: 4
867: 2
868: 3
869: 3
870: 2
871: 3
872: 2
873: 2
874: 2
875: 2
876: 2
877: 2
878: 5
879: 3
880: 5
881: 1
882: 5
883: 5
884: 5
885: 5
886: 5
887: 5
888: 1
889: 4
890: 5
891: 2
892: 5
893: 3
894: 2
895: 4
896: 3
897: 1
898: 2
899: 2
900: 2
901: 4
902: 3
903: 2
904: 3
905: 3
906: 4
907: 2
908: 5
909: 2
910: 1
911: 4
912: 3
913: 2
914: 2
915: 2
916: 3
917: 3
918: 2
919: 5
920: 2
921: 5
922: 2
923: 2
924: 2
925: 3
926: 3
927: 2
928: 3
929: 2
930: 2
931: 2
932: 2
933: 2
934: 2
935: 2
936: 2
937: 2
938: 2
939: 3
940: 2
941: 3
942: 2
943: 2
944: 2
945: 2
946: 2
947: 3
948: 2
949: 3
950: 2
951: 2
952: 5
953: 5
954: 5
955: 1
956: 2
957: 3
958: 2
959: 2
960: 5
961: 5
962: 2
963: 3
964: 2
965: 2
966: 2
967: 3
968: 2
969: 2
970: 5
971: 2
972: 1
973: 2
974: 1
975: 2
976: 3
977: 1
978: 3
979: 2
980: 2
981: 3
982: 3
983: 2
984: 2
985: 4
986: 2
987: 3
988: 2
989: 3
990: 2
991: 2
992: 2
993: 3
994: 3
995: 3
996: 3
997: 3
998: 3
999: 3
1000: 3
1001: 3
1002: 3
1003: 3
1004: 2
1005: 3
1006: 2
1007: 3
1008: 3
1009: 4
1010: 5
1011: 2
1012: 3
1013: 3
1014: 2
1015: 4
1016: 2
1017: 1
1018: 3
1019: 3
1020: 2
1021: 4
1022: 3
1023: 3
1024: 2
1025: 2
1026: 1
1027: 2
1028: 1
1029: 2
1030: 2
1031: 2
1032: 3
1033: 3
1034: 2
1035: 3
1036: 3
1037: 5
1038: 3
1039: 3
1040: 3
1041: 2
1042: 4
1043: 3
1044: 3
1045: 3
1046: 3
1047: 3
1048: 3
1049: 3
1050: 2
1051: 2
1052: 3
1053: 5
1054: 2
1055: 1
1056: 2
1057: 2
1058: 3
1059: 4
1060: 3
1061: 2
1062: 3
1063: 2
1064: 3
1065: 4
1066: 3
1067: 2
1068: 2
1069: 3
1070: 2
1071: 2
1072: 1
1073: 2
1074: 2
1075: 2
1076: 2
1077: 3
1078: 2
1079: 2
1080: 2
1081: 2
1082: 3
1083: 3
1084: 3
1085: 5
1086: 5
1087: 3
1088: 2
1089: 3
1090: 3
1091: 3
1092: 3
1093: 2
1094: 3
1095: 3
1096: 5
1097: 4
1098: 3
1099: 3
1100: 3
1101: 3
1102: 4
1103: 3
1104: 3
1105: 3
1106: 3
1107: 3
1108: 3
1109: 3
1110: 3
1111: 3
1112: 4
1113: 3
1114: 3
1115: 2
1116: 3
1117: 2
1118: 2
1119: 3
1120: 2
1121: 3
1122: 4
1123: 3
1124: 4
1125: 3
1126: 3
1127: 2
1128: 2
1129: 3
1130: 4
1131: 2
1132: 2
1133: 3
1134: 2
1135: 3
1136: 2
1137: 1
1138: 2
1139: 1
1140: 2
1141: 1
1142: 3
1143: 2
1144: 3
1145: 2
1146: 1
1147: 2
1148: 3
1149: 3
1150: 3
1151: 2
1152: 3
1153: 2
1154: 2
1155: 3
1156: 2
1157: 2
1158: 2
1159: 3
1160: 3
1161: 2
1162: 2
1163: 2
1164: 4
1165: 3
1166: 2
1167: 2
1168: 3
1169: 3
1170: 4
1171: 3
1172: 3
1173: 3
1174: 3
1175: 2
"""

# Parse grades using regular expression
grade_pattern = re.compile(r'(?:ID:?\s*)?(\d+)\s*[\s,:-]+\s*([0-5])', re.IGNORECASE)
parsed_grades = {}

for line in ai_output_raw.strip().split('\n'):
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    match = grade_pattern.search(line)
    if match:
        item_id = int(match.group(1))
        score = int(match.group(2))
        parsed_grades[item_id] = score

print(f"Parsed {len(parsed_grades)} grades from the AI output.")

if len(parsed_grades) > 0:
    pool_df['relevance'] = pool_df['id'].map(parsed_grades)
    missing_count = pool_df['relevance'].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} rows were not graded. Setting their relevance to 0.")
        pool_df['relevance'] = pool_df['relevance'].fillna(0)
    else:
        print("All rows successfully graded!")
        
    pool_df['relevance'] = pool_df['relevance'].astype(int)
    output_df = pool_df[['query', 'movie_title', 'relevance']].copy()
    
    output_csv = '../data/qrels_to_grade.csv'
    output_df.to_csv(output_csv, index=False)
    print(f"Successfully exported final graded qrels to '{output_csv}' with {len(output_df)} rows.")
else:
    print("No grades were parsed. Please paste the AI output in 'ai_output_raw' or write to 'ai_grades.txt' and rerun this cell.")

Parsed 1176 grades from the AI output.
All rows successfully graded!
Successfully exported final graded qrels to '../data/qrels_to_grade.csv' with 1176 rows.
